# Isochrone age sampling

This tutorial shows how to choose a coeval isochrone age before converting a population draw into stars for synthetic-spectrum rendering. The examples use a tiny in-memory isochrone bank so the notebook can run without external data.

In [ ]:
import numpy as np

from minato.synthetic import (
    BinarySystem,
    IsochroneAgeSampler,
    IsochronePoint,
    LoggSkewWeight,
    StellarConstraints,
)

A real workflow can use `IsochroneBank` with CSV files. For the tutorial, this small class implements the same `ages` and `interpolate(mass, log_age)` interface.

In [ ]:
class ToyIsochroneBank:
    def __init__(self, tables):
        self.tables = {float(age): table for age, table in tables.items()}
        self.ages = np.array(sorted(self.tables), dtype=float)

    def interpolate(self, mass, log_age):
        age_matches = self.ages[np.isclose(self.ages, float(log_age))]
        if age_matches.size == 0:
            raise ValueError(f"log_age {log_age} is not in this toy bank")
        table = self.tables[float(age_matches[0])]
        masses = table["mass_init"]
        if mass < masses[0] or mass > masses[-1]:
            raise ValueError(f"mass {mass} is outside this slice range")
        return IsochronePoint(
            teff=float(np.interp(mass, masses, table["teff"])),
            logg=float(np.interp(mass, masses, table["logg"])),
            radius=float(np.interp(mass, masses, table["radius"])),
        )


def table(mass_init, teff, logg, radius):
    return {
        "mass_init": np.array(mass_init, dtype=float),
        "teff": np.array(teff, dtype=float),
        "logg": np.array(logg, dtype=float),
        "radius": np.array(radius, dtype=float),
    }


iso = ToyIsochroneBank(
    {
        6.80: table(
            [4.0, 8.0, 19.0, 20.0, 25.0],
            [14_000.0, 22_000.0, 33_500.0, 34_000.0, 36_000.0],
            [4.45, 4.30, 4.12, 4.10, 3.95],
            [2.4, 3.8, 6.8, 7.2, 8.5],
        ),
        6.90: table(
            [15.0, 19.0, 20.0, 25.0],
            [28_000.0, 32_000.0, 33_000.0, 35_000.0],
            [4.20, 4.02, 3.95, 3.70],
            [5.2, 7.5, 8.0, 10.0],
        ),
        7.00: table(
            [4.0, 8.0, 19.0, 20.0, 25.0],
            [7_500.0, 8_000.0, 9_000.0, 9_200.0, 9_500.0],
            [2.90, 2.80, 2.50, 2.45, 2.30],
            [18.0, 24.0, 34.0, 36.0, 40.0],
        ),
        7.10: table(
            [4.0, 8.0, 19.0, 20.0, 25.0],
            [12_000.0, 18_000.0, 25_000.0, 26_000.0, 28_000.0],
            [4.25, 4.05, 3.88, 3.85, 3.60],
            [3.0, 5.0, 9.0, 9.5, 11.0],
        ),
    }
)

## Generic main-sequence selection

The default sampler has no temperature or gravity cuts. Add only the constraints that matter for the population-to-spectrum mapping you are doing.

In [ ]:
rng = np.random.default_rng(10)

main_sequence = IsochroneAgeSampler(
    primary=StellarConstraints(logg_min=3.5),
    secondary=StellarConstraints(logg_min=3.5),
)

log_age, age_meta = main_sequence.sample(iso, m1=8.0, m2=5.0, rng=rng)
log_age, age_meta["valid_log_ages"], age_meta["selected_primary"]

## OB / hot-star selection

Hot-star cuts are not MINATO defaults. They are configured explicitly here so an evolved, cool low-gravity age slice is rejected before spectra are rendered.

In [ ]:
hot_star = IsochroneAgeSampler(
    primary=StellarConstraints(teff_min=10_000.0, logg_min=3.0),
    secondary=StellarConstraints(logg_min=3.0, logg_max=5.5),
    secondary_low_mass_teff_max={"mass_max": 8.0, "teff_max": 20_000.0},
    require_primary_logg_lte_secondary=True,
    weight=LoggSkewWeight(mu=4.0, sigma_lo=0.25, sigma_hi=0.12),
)

log_age, age_meta = hot_star.sample(
    iso,
    m1=19.28,
    m2=19.05,
    rng=np.random.default_rng(123),
)

log_age, age_meta["valid_log_ages"], age_meta["selected_primary"], age_meta["selected_secondary"]

The sampled age can then be passed into the existing constructors, or the convenience constructor can do that explicit sampling step for a binary.

In [ ]:
system = BinarySystem.from_masses_with_age_sampler(
    19.28,
    19.05 / 19.28,
    iso,
    hot_star,
    rng=np.random.default_rng(123),
)

system.metadata["log_age"], system.primary.teff, system.secondary.teff